In [2]:
import re
import json
import unicodedata
from pathlib import Path

import pandas as pd


In [3]:
DATA_PATH = Path(r"E:\Semester6\Laptrinhpython\project\data\formatted_jobs.csv")

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(970, 6)


,ID_num,job_title,Short_description,Skills_required,Industry,Pay_grade
0,1,Software Engineer,Develop and maintain web applications using mo...,Problem Solving Logical Reasoning Attention to...,Technology,High paying
1,2,Data Scientist,Analyze large datasets to extract business ins...,Analytical Thinking Pattern Recognition Mathem...,Technology,High paying
2,3,Marketing Manager,Lead marketing campaigns and brand strategy de...,Creative Thinking Strategic Planning Communica...,Marketing,Average paying
3,4,UX Designer,Design user-friendly interfaces and improve us...,Creative Problem Solving Empathy Research Skil...,Technology,Average paying
4,5,Financial Analyst,Analyze financial data and prepare reports for...,Analytical Thinking Attention to Detail Mathem...,Finance,Average paying


In [4]:
required_cols = ["job_title", "Skills_required"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Thiếu cột: {missing_cols}")

df = df[required_cols].copy()
df.head()

,job_title,Skills_required
0,Software Engineer,Problem Solving Logical Reasoning Attention to...
1,Data Scientist,Analytical Thinking Pattern Recognition Mathem...
2,Marketing Manager,Creative Thinking Strategic Planning Communica...
3,UX Designer,Creative Problem Solving Empathy Research Skil...
4,Financial Analyst,Analytical Thinking Attention to Detail Mathem...


In [5]:
SPECIAL_TOKEN_MAP = {
    "c++": "cplusplus",
    "c#": "csharp",
    ".net": "dotnet",
    "asp.net": "aspdotnet",
    "node.js": "nodejs",
    "node js": "nodejs",
    "react.js": "reactjs",
    "vue.js": "vuejs",
    "next.js": "nextjs",
    "sql server": "sqlserver",
    "power bi": "powerbi",
    "machine learning": "machinelearning",
    "deep learning": "deeplearning",
    "data analyst": "dataanalyst",
    "data scientist": "datascientist",
    "data engineer": "dataengineer",
    "software engineer": "softwareengineer",
    "software developer": "softwaredeveloper",
    "backend developer": "backenddeveloper",
    "frontend developer": "frontenddeveloper",
    "full stack": "fullstack",
    "project manager": "projectmanager",
    "product manager": "productmanager",
}


def normalize_text_basic(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\x00", " ").replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def protect_special_tokens(text: str) -> str:
    text = text.lower()
    for src, dst in sorted(SPECIAL_TOKEN_MAP.items(), key=lambda x: len(x[0]), reverse=True):
        text = re.sub(re.escape(src), dst, text)
    return text


def restore_special_tokens(text: str) -> str:
    reverse_map = {v: k for k, v in SPECIAL_TOKEN_MAP.items()}
    for src, dst in sorted(reverse_map.items(), key=lambda x: len(x[0]), reverse=True):
        text = re.sub(rf"\b{re.escape(src)}\b", dst, text)
    return text


In [6]:
def clean_job_title(text: str) -> str:
    text = normalize_text_basic(text)
    if not text:
        return ""

    text = protect_special_tokens(text)

    text = text.lower()

    # bỏ ngoặc và ký hiệu nhiễu
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"\[[^\]]*\]", " ", text)

    # chỉ giữ chữ, số, khoảng trắng và vài ký tự cơ bản
    text = re.sub(r"[^a-z0-9\s\-_/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    text = restore_special_tokens(text)

    return text


In [7]:
def split_skill_items(text: str) -> list[str]:
    text = normalize_text_basic(text)
    if not text:
        return []

    # chuẩn hóa separator
    text = text.replace("•", ";")
    text = text.replace("|", ";")
    text = text.replace("\n", ";")
    text = text.replace("\r", ";")

    # tách theo ; hoặc ,
    raw_items = re.split(r"[;,]", text)

    items = []
    for item in raw_items:
        item = item.strip()
        if item:
            items.append(item)

    return items


def clean_skill(text: str) -> str:
    text = normalize_text_basic(text)
    if not text:
        return ""

    text = protect_special_tokens(text)
    text = text.lower()

    # bỏ ngoặc
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"\[[^\]]*\]", " ", text)

    # chỉ giữ chữ, số, khoảng trắng và vài ký tự cơ bản
    text = re.sub(r"[^a-z0-9\s\-_/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    text = restore_special_tokens(text)

    return text


In [9]:
title_set = set()

for title in df["job_title"].dropna():
    cleaned = clean_job_title(title)
    if cleaned:
        title_set.add(cleaned)

title_dict = sorted(title_set)

print("Số lượng title:", len(title_dict))
print(title_dict[:50])


Số lượng title: 896
['3d printing technician', 'accountant', 'accounts receivable specialist', 'actor', 'administrative assistant', 'advertising executive', 'aerial photography pilot', 'aerospace engineer', 'ai chatbot designer', 'ai data labeling supervisor', 'ai ethics consultant', 'ai model validator', 'ai prompt engineer', 'ai research scientist', 'ai-driven content strategist', 'ai-powered marketing analyst', 'air traffic controller', 'amusement park attendant', 'animal behaviorist', 'animal kennel worker', 'animation artist', 'anthropologist', 'application support analyst', 'aquaculture farmer', 'aquaponics farmer', 'aquarium maintenance specialist', 'architect', 'archivist', 'art director', 'artificial intelligence trainer', 'artisanal bread baker', 'artisanal chocolate maker', 'artisanal fermented foods producer', 'artisanal fiber artist and dyer', 'artisanal ice cream maker', 'artisanal pickle and preserve maker', 'artisanal soap and cosmetics maker', 'assembly line worker', '

In [8]:
skill_set = set()

for skill_cell in df["Skills_required"].dropna():
    items = split_skill_items(skill_cell)

    for item in items:
        cleaned = clean_skill(item)
        if cleaned:
            skill_set.add(cleaned)

skill_dict = sorted(skill_set)

print("Số lượng skill:", len(skill_dict))
print(skill_dict[:100])


Số lượng skill: 1120
['3d modeling programming optical physics creative vision technical innovation', '3d printing', '3d programming user experience design spatial design innovation technical skills', 'a/b testing', 'accessibility', 'accessibility standards', 'accessibility testing', 'account management', 'accounting', 'accounting software', 'accounting standards', 'accounts receivable', 'accuracy', 'active listening', 'active listening empathy analytical thinking communication ethical reasoning', 'active listening patience problem solving communication empathy', 'administrative support', 'administrative tasks', 'adobe creative suite', 'adult learning', 'advanced analytics', 'advanced programming', 'advanced statistics', 'advertising', 'advocacy', 'aerodynamics', 'aerospace engineering aerodynamics innovation problem solving technical skills', 'aerospace engineering materials science physics innovation problem solving', 'aerospace engineering mining engineering space science innovation

In [10]:
def is_valid_title_term(term: str) -> bool:
    if not term:
        return False
    if len(term) < 2:
        return False
    if len(term.split()) > 8:
        return False
    if re.fullmatch(r"\d+", term):
        return False
    return True


def is_valid_skill_term(term: str) -> bool:
    if not term:
        return False
    if len(term) < 2:
        return False
    if len(term.split()) > 6:
        return False
    if re.fullmatch(r"\d+", term):
        return False
    return True


title_dict = [t for t in title_dict if is_valid_title_term(t)]
skill_dict = [s for s in skill_dict if is_valid_skill_term(s)]

print("Title sau lọc:", len(title_dict))
print("Skill sau lọc:", len(skill_dict))


Title sau lọc: 896
Skill sau lọc: 640


In [11]:
print("=== TITLE SAMPLE ===")
print(title_dict[:50])

print("\n=== TITLE SAMPLE END ===")
print(title_dict[-50:])

print("\n=== SKILL SAMPLE ===")
print(skill_dict[:100])

print("\n=== SKILL SAMPLE END ===")
print(skill_dict[-100:])


=== TITLE SAMPLE ===
['3d printing technician', 'accountant', 'accounts receivable specialist', 'actor', 'administrative assistant', 'advertising executive', 'aerial photography pilot', 'aerospace engineer', 'ai chatbot designer', 'ai data labeling supervisor', 'ai ethics consultant', 'ai model validator', 'ai prompt engineer', 'ai research scientist', 'ai-driven content strategist', 'ai-powered marketing analyst', 'air traffic controller', 'amusement park attendant', 'animal behaviorist', 'animal kennel worker', 'animation artist', 'anthropologist', 'application support analyst', 'aquaculture farmer', 'aquaponics farmer', 'aquarium maintenance specialist', 'architect', 'archivist', 'art director', 'artificial intelligence trainer', 'artisanal bread baker', 'artisanal chocolate maker', 'artisanal fermented foods producer', 'artisanal fiber artist and dyer', 'artisanal ice cream maker', 'artisanal pickle and preserve maker', 'artisanal soap and cosmetics maker', 'assembly line worker', 

In [12]:
output_dir = Path(r"E:\Semester6\Laptrinhpython\project\data")
output_dir.mkdir(parents=True, exist_ok=True)

title_output_path = output_dir / "title_dict.json"
skill_output_path = output_dir / "skill_dict.json"

with open(title_output_path, "w", encoding="utf-8") as f:
    json.dump(title_dict, f, ensure_ascii=False, indent=2)

with open(skill_output_path, "w", encoding="utf-8") as f:
    json.dump(skill_dict, f, ensure_ascii=False, indent=2)

print("Đã lưu:")
print(title_output_path)
print(skill_output_path)


Đã lưu:
E:\Semester6\Laptrinhpython\project\data\title_dict.json
E:\Semester6\Laptrinhpython\project\data\skill_dict.json


In [13]:
pd.DataFrame({"title": title_dict}).to_csv(
    output_dir / "title_dict.csv", index=False, encoding="utf-8"
)

pd.DataFrame({"skill": skill_dict}).to_csv(
    output_dir / "skill_dict.csv", index=False, encoding="utf-8"
)

print("Đã lưu thêm CSV.")

Đã lưu thêm CSV.


In [14]:
with open(title_output_path, "r", encoding="utf-8") as f:
    loaded_title_dict = json.load(f)

with open(skill_output_path, "r", encoding="utf-8") as f:
    loaded_skill_dict = json.load(f)

print(len(loaded_title_dict), len(loaded_skill_dict))
print(loaded_title_dict[:10])
print(loaded_skill_dict[:20])


896 640
['3d printing technician', 'accountant', 'accounts receivable specialist', 'actor', 'administrative assistant', 'advertising executive', 'aerial photography pilot', 'aerospace engineer', 'ai chatbot designer', 'ai data labeling supervisor']
['3d printing', 'a/b testing', 'accessibility', 'accessibility standards', 'accessibility testing', 'account management', 'accounting', 'accounting software', 'accounting standards', 'accounts receivable', 'accuracy', 'active listening', 'administrative support', 'administrative tasks', 'adobe creative suite', 'adult learning', 'advanced analytics', 'advanced programming', 'advanced statistics', 'advertising']
